# PyTorch: From Basics to Advanced

**Purpose:** Build fluency in PyTorch so you can read and understand deep learning code in ML/AI projects.

PyTorch is the dominant framework for deep learning research and increasingly for production. If you see a neural network in a paper or open-source project, it's most likely PyTorch.

**How this notebook is organized:**
1. Tensors — Creating, Properties, and dtypes
2. Tensor Operations — Math, Indexing, and Reshaping
3. NumPy ↔ PyTorch Interop
4. Autograd — Automatic Differentiation
5. Neural Network Building Blocks (`nn.Module`)
6. Loss Functions and Optimizers
7. The Training Loop (the heart of every DL project)
8. Data Loading (`Dataset` and `DataLoader`)
9. Saving and Loading Models
10. GPU / Device Management
11. Common Architectures (CNN, RNN patterns)
12. Real Patterns You'll See in AI Code

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # Fix OpenMP conflict on Windows

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cpu
CUDA available:  False


## 1. Tensors — Creating, Properties, and dtypes

A **tensor** is PyTorch's version of a NumPy array. The key difference: tensors can live on GPUs and track gradients for backpropagation.

```
NumPy:   np.array    →  ndarray
PyTorch: torch.tensor → Tensor
```

In [2]:
# From Python lists — same as np.array()
a = torch.tensor([1, 2, 3, 4, 5])
print("1D tensor:", a)

# 2D tensor (matrix)
m = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print("\n2D tensor:\n", m)

# With explicit dtype
floats = torch.tensor([1, 2, 3], dtype=torch.float32)
print("\nFloat32:", floats)

1D tensor: tensor([1, 2, 3, 4, 5])

2D tensor:
 tensor([[1, 2, 3],
        [4, 5, 6]])

Float32: tensor([1., 2., 3.])


In [3]:
# Factory functions — mirrors NumPy exactly
zeros = torch.zeros(3, 4)          # np.zeros((3, 4))
ones = torch.ones(2, 3)            # np.ones((2, 3))
full = torch.full((2, 2), 7.0)     # np.full((2, 2), 7.0)
eye = torch.eye(3)                 # np.eye(3)
arange = torch.arange(0, 10, 2)    # np.arange(0, 10, 2)
linspace = torch.linspace(0, 1, 5) # np.linspace(0, 1, 5)

print("zeros:\n", zeros)
print("arange:", arange)
print("linspace:", linspace)

# Random tensors — THE most common in ML code
rand = torch.rand(3, 4)        # Uniform [0, 1) — like np.random.rand
randn = torch.randn(3, 4)      # Normal N(0,1)  — like np.random.randn
randint = torch.randint(0, 10, (3, 4))  # Random integers

print("\nrandn (3x4):\n", randn)

zeros:
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])

randn (3x4):
 tensor([[ 0.1066, -1.4947, -1.9999, -1.4806],
        [-0.3247,  0.1092,  0.5577,  0.5960],
        [-0.2799, -0.3882, -0.3235, -0.8355]])


In [4]:
# Tensor properties — same concepts as NumPy
t = torch.randn(3, 4)

print("shape:", t.shape)       # or t.size() — both work
print("dtype:", t.dtype)       # torch.float32 is the default (NumPy uses float64)
print("ndim: ", t.ndim)        # number of dimensions
print("numel:", t.numel())     # total elements (like np.size)

# Common dtypes in PyTorch:
# torch.float32 (default) — model weights, features, outputs
# torch.float16           — mixed precision training (faster on GPU)
# torch.int64 (torch.long) — class labels, indices
# torch.bool              — masks

# Type conversion
a = torch.tensor([1, 2, 3])
print("\nOriginal:", a.dtype)
print("To float:", a.float().dtype)     # .float() is shorthand for .to(torch.float32)
print("To long: ", a.long().dtype)      # .long() is shorthand for .to(torch.int64)
print("To bool: ", a.bool())

shape: torch.Size([3, 4])
dtype: torch.float32
ndim:  2
numel: 12

Original: torch.int64
To float: torch.float32
To long:  torch.int64
To bool:  tensor([True, True, True])


## 2. Tensor Operations — Math, Indexing, and Reshaping

Nearly identical to NumPy. If you know NumPy, you already know 90% of PyTorch tensor ops.

In [5]:
# Element-wise math — same as NumPy
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([10.0, 20.0, 30.0, 40.0])

print("a + b:  ", a + b)
print("a * b:  ", a * b)       # Element-wise, NOT matrix multiply
print("a ** 2: ", a ** 2)
print("a + 10: ", a + 10)      # Scalar broadcast

# Common math functions
print("\nsqrt:  ", torch.sqrt(a))
print("exp:   ", torch.exp(a))
print("log:   ", torch.log(a))
print("clamp: ", torch.clamp(a, min=1.5, max=3.5))  # Like np.clip

a + b:   tensor([11., 22., 33., 44.])
a * b:   tensor([ 10.,  40.,  90., 160.])
a ** 2:  tensor([ 1.,  4.,  9., 16.])
a + 10:  tensor([11., 12., 13., 14.])

sqrt:   tensor([1.0000, 1.4142, 1.7321, 2.0000])
exp:    tensor([ 2.7183,  7.3891, 20.0855, 54.5981])
log:    tensor([0.0000, 0.6931, 1.0986, 1.3863])
clamp:  tensor([1.5000, 2.0000, 3.0000, 3.5000])


In [6]:
# Aggregation — same as NumPy but uses dim= instead of axis=
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("sum:     ", t.sum())
print("mean:    ", t.mean())
print("max:     ", t.max())
print("argmax:  ", t.argmax())         # Index of max in flattened tensor

# dim= is PyTorch's version of NumPy's axis=
# dim=0: collapse rows (operate DOWN columns)
# dim=1: collapse columns (operate ACROSS rows)
print("\nsum dim=0:", t.sum(dim=0))     # [5, 7, 9]
print("sum dim=1:", t.sum(dim=1))       # [6, 15]
print("mean dim=0:", t.mean(dim=0))

# max returns (values, indices) when dim is specified
vals, idxs = t.max(dim=1)
print("\nmax dim=1 values:", vals)
print("max dim=1 indices:", idxs)

sum:      tensor(21.)
mean:     tensor(3.5000)
max:      tensor(6.)
argmax:   tensor(5)

sum dim=0: tensor([5., 7., 9.])
sum dim=1: tensor([ 6., 15.])
mean dim=0: tensor([2.5000, 3.5000, 4.5000])

max dim=1 values: tensor([3., 6.])
max dim=1 indices: tensor([2, 2])


In [7]:
# Indexing and slicing — IDENTICAL to NumPy
m = torch.tensor([[1,  2,  3,  4],
                  [5,  6,  7,  8],
                  [9, 10, 11, 12]])

print("Element [1,2]:", m[1, 2])          # 7
print("Row 0:        ", m[0])              # [1, 2, 3, 4]
print("Column 1:     ", m[:, 1])           # [2, 6, 10]
print("Submatrix:\n",   m[0:2, 1:3])      # rows 0-1, cols 1-2

# Boolean indexing — same as NumPy
a = torch.tensor([10, 25, 5, 30, 15])
print("\na > 12:  ", a[a > 12])            # [25, 30, 15]
print("where:   ", torch.where(a > 12, a, torch.zeros_like(a)))  # Like np.where

Element [1,2]: tensor(7)
Row 0:         tensor([1, 2, 3, 4])
Column 1:      tensor([ 2,  6, 10])
Submatrix:
 tensor([[2, 3],
        [6, 7]])

a > 12:   tensor([25, 30, 15])
where:    tensor([ 0, 25,  0, 30, 15])


In [8]:
# Reshaping — the most critical operations in deep learning
a = torch.arange(12)

# reshape — same as NumPy, -1 means "infer this dimension"
print("reshape(3,4):\n", a.reshape(3, 4))
print("reshape(-1,3):", a.reshape(-1, 3).shape)  # (4, 3)

# view — same as reshape but ALWAYS returns a view (shared memory)
# You'll see .view() more often than .reshape() in PyTorch code
print("\nview(3,4):\n", a.view(3, 4))
print("view(-1,3):", a.view(-1, 3).shape)

# unsqueeze / squeeze — add or remove dimensions of size 1
# CRITICAL for matching tensor shapes in neural networks
t = torch.tensor([1, 2, 3])             # shape: (3,)
print("\nOriginal shape:", t.shape)
print("unsqueeze(0):  ", t.unsqueeze(0).shape)   # (1, 3) — add batch dim
print("unsqueeze(1):  ", t.unsqueeze(1).shape)   # (3, 1) — add feature dim

t2 = torch.randn(1, 3, 1)
print("\nsqueeze:       ", t2.squeeze().shape)    # Removes ALL dims of size 1 → (3,)
print("squeeze(0):    ", t2.squeeze(0).shape)     # Removes only dim 0 → (3, 1)

reshape(3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
reshape(-1,3): torch.Size([4, 3])

view(3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
view(-1,3): torch.Size([4, 3])

Original shape: torch.Size([3])
unsqueeze(0):   torch.Size([1, 3])
unsqueeze(1):   torch.Size([3, 1])

squeeze:        torch.Size([3])
squeeze(0):     torch.Size([3, 1])


In [9]:
# More reshaping ops you'll see constantly

# Transpose
m = torch.randn(2, 3)
print("Transpose .T:     ", m.T.shape)            # (3, 2) — shorthand
print("transpose(0,1):   ", m.transpose(0, 1).shape)  # Same thing, explicit

# Permute — reorder arbitrary dimensions (used in image processing)
# Images: (batch, channels, height, width) ↔ (batch, height, width, channels)
img = torch.randn(8, 3, 32, 32)   # batch=8, C=3, H=32, W=32
print("\nOriginal image batch:", img.shape)
print("permute(0,2,3,1):   ", img.permute(0, 2, 3, 1).shape)  # → (8, 32, 32, 3)

# Flatten — collapse dimensions (before feeding to linear layer)
print("\nflatten:            ", img.flatten(1).shape)   # (8, 3072) — flatten all except batch

# Contiguous — some operations require contiguous memory
# You'll see .contiguous() after transpose/permute
t = m.T.contiguous()

# cat and stack — combining tensors
a = torch.randn(2, 3)
b = torch.randn(2, 3)
print("\ncat dim=0:", torch.cat([a, b], dim=0).shape)    # (4, 3) — stack rows
print("cat dim=1:", torch.cat([a, b], dim=1).shape)      # (2, 6) — stack cols
print("stack:    ", torch.stack([a, b], dim=0).shape)     # (2, 2, 3) — new dim

Transpose .T:      torch.Size([3, 2])
transpose(0,1):    torch.Size([3, 2])

Original image batch: torch.Size([8, 3, 32, 32])
permute(0,2,3,1):    torch.Size([8, 32, 32, 3])

flatten:             torch.Size([8, 3072])

cat dim=0: torch.Size([4, 3])
cat dim=1: torch.Size([2, 6])
stack:     torch.Size([2, 2, 3])


In [10]:
# Matrix multiplication — the core computation in neural networks
# Three ways (all equivalent for 2D):
a = torch.randn(3, 4)
b = torch.randn(4, 5)

c1 = a @ b                    # Operator (most common)
c2 = torch.matmul(a, b)       # Function
c3 = torch.mm(a, b)           # 2D-only function

print("a @ b shape:", c1.shape)     # (3, 5)
print("All equal:  ", torch.allclose(c1, c2) and torch.allclose(c1, c3))

# Batch matrix multiply — matmul handles this automatically
# (batch, m, k) @ (batch, k, n) → (batch, m, n)
batch_a = torch.randn(8, 3, 4)  # 8 matrices of shape 3x4
batch_b = torch.randn(8, 4, 5)  # 8 matrices of shape 4x5
result = batch_a @ batch_b
print("\nBatch matmul:", result.shape)  # (8, 3, 5)

a @ b shape: torch.Size([3, 5])
All equal:   True

Batch matmul: torch.Size([8, 3, 5])


## 3. NumPy ↔ PyTorch Interop

You'll constantly convert between NumPy and PyTorch, especially when loading data from pandas/sklearn.

In [11]:
# NumPy → PyTorch
np_array = np.array([1.0, 2.0, 3.0])
tensor = torch.from_numpy(np_array)     # Shares memory! (like a view)
tensor2 = torch.tensor(np_array)        # Copies data (independent)
print("from_numpy:", tensor)
print("tensor():  ", tensor2)

# PyTorch → NumPy
t = torch.tensor([4.0, 5.0, 6.0])
np_back = t.numpy()                     # Shares memory
np_copy = t.detach().numpy()            # Safe copy (detach from grad tracking)
print("\n.numpy():  ", np_back)

# Common pattern: pandas → numpy → tensor
# X_tensor = torch.tensor(df[feature_cols].values, dtype=torch.float32)
# y_tensor = torch.tensor(df['target'].values, dtype=torch.long)

# WARNING: shared memory means changes propagate!
np_array[0] = 999
print("\nAfter modifying NumPy, from_numpy tensor:", tensor)   # Also changed!
print("tensor() copy:                            ", tensor2)   # Unchanged

from_numpy: tensor([1., 2., 3.], dtype=torch.float64)
tensor():   tensor([1., 2., 3.], dtype=torch.float64)

.numpy():   [4. 5. 6.]

After modifying NumPy, from_numpy tensor: tensor([999.,   2.,   3.], dtype=torch.float64)
tensor() copy:                             tensor([1., 2., 3.], dtype=torch.float64)


## 4. Autograd — Automatic Differentiation

**This is what makes PyTorch special.** Autograd tracks all operations on tensors and automatically computes gradients (derivatives) for backpropagation. This is how neural networks learn.

In [12]:
# requires_grad=True tells PyTorch: "track all operations on this tensor"
x = torch.tensor([2.0, 3.0], requires_grad=True)

# Forward pass: compute y = x^2 + 3x
y = x ** 2 + 3 * x
print("x:", x)
print("y:", y)

# Backward pass: compute gradients (dy/dx)
# Need a scalar for .backward(), so sum first
loss = y.sum()
loss.backward()  # Computes gradients and stores them in x.grad

# dy/dx = 2x + 3
# At x=[2,3]: gradients should be [7, 9]
print("\nGradients (dy/dx):", x.grad)  # [7.0, 9.0] ✓

# IMPORTANT: gradients accumulate by default! Must zero them between iterations
# x.grad.zero_()  — the underscore means "in-place"

x: tensor([2., 3.], requires_grad=True)
y: tensor([10., 18.], grad_fn=<AddBackward0>)

Gradients (dy/dx): tensor([7., 9.])


In [13]:
# torch.no_grad() — disable gradient tracking
# You'll see this in EVERY PyTorch project during evaluation/inference

x = torch.randn(3, requires_grad=True)

# During training: gradients are tracked
y = x * 2
print("requires_grad during training:", y.requires_grad)  # True

# During evaluation: no need to track gradients (saves memory and speeds up)
with torch.no_grad():
    y = x * 2
    print("requires_grad during eval:    ", y.requires_grad)  # False

# .detach() — removes a tensor from the computation graph
# Used when you want to use a tensor's value without tracking its gradient
z = x.detach()
print("\ndetached requires_grad:", z.requires_grad)  # False

# You'll see these patterns:
# with torch.no_grad():     — wraps evaluation/inference code
# tensor.detach()           — when converting to numpy or using as a constant
# tensor.detach().cpu().numpy() — THE pattern for getting values out of PyTorch

requires_grad during training: True
requires_grad during eval:     False

detached requires_grad: False


## 5. Neural Network Building Blocks (`nn.Module`)

`nn.Module` is the base class for ALL neural networks in PyTorch. Every model you'll ever read inherits from it.

In [14]:
# Common layers you'll encounter — understand these and you can read most models

# Linear (fully connected / dense) layer
# y = x @ W.T + b
linear = nn.Linear(in_features=10, out_features=5)
x = torch.randn(32, 10)    # batch of 32, each with 10 features
out = linear(x)
print("Linear: input", x.shape, "→ output", out.shape)  # (32, 5)

# The layer has learnable parameters
print("Weight shape:", linear.weight.shape)   # (5, 10)
print("Bias shape:  ", linear.bias.shape)     # (5,)

Linear: input torch.Size([32, 10]) → output torch.Size([32, 5])
Weight shape: torch.Size([5, 10])
Bias shape:   torch.Size([5])


In [15]:
# Activation functions — introduce non-linearity
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

# ReLU — the default choice for hidden layers
print("ReLU:    ", F.relu(x))          # max(0, x) → [0, 0, 0, 1, 2]

# Sigmoid — binary classification output
print("Sigmoid: ", torch.sigmoid(x))    # Squashes to [0, 1]

# Softmax — multi-class classification output
logits = torch.tensor([2.0, 1.0, 0.1])
print("Softmax: ", F.softmax(logits, dim=0))  # Sums to 1.0

# Tanh — sometimes used in RNNs
print("Tanh:    ", torch.tanh(x))       # Squashes to [-1, 1]

# You'll see activations used two ways:
# 1. As functions:  F.relu(x)            — in the forward() method
# 2. As layers:     nn.ReLU()            — in __init__, stored as module
# Both are equivalent; functions are slightly more common for simple activations

ReLU:     tensor([0., 0., 0., 1., 2.])
Sigmoid:  tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
Softmax:  tensor([0.6590, 0.2424, 0.0986])
Tanh:     tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])


In [16]:
# Other common layers you'll encounter:

# Dropout — randomly zeros elements during training (prevents overfitting)
dropout = nn.Dropout(p=0.5)  # 50% of values become 0
x = torch.ones(10)
print("Dropout (training): ", dropout(x))        # Some values are 0
dropout.eval()  # Switch to eval mode
print("Dropout (eval):     ", dropout(x))        # All values pass through

# BatchNorm — normalizes layer outputs (stabilizes training)
bn = nn.BatchNorm1d(num_features=5)
x = torch.randn(32, 5)  # batch of 32, 5 features
print("\nBatchNorm shape:", bn(x).shape)  # Same shape, but normalized

# Embedding — converts integer IDs to dense vectors (NLP, recommendations)
embed = nn.Embedding(num_embeddings=1000, embedding_dim=64)  # 1000 words, 64-dim vectors
word_ids = torch.tensor([5, 42, 999, 0])
print("\nEmbedding:", embed(word_ids).shape)  # (4, 64) — 4 words → 4 vectors

Dropout (training):  tensor([2., 2., 2., 2., 2., 0., 0., 2., 0., 2.])
Dropout (eval):      tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

BatchNorm shape: torch.Size([32, 5])



Embedding: torch.Size([4, 64])


In [17]:
# Building a model — THE pattern you'll see in every PyTorch project
# Step 1: Define the model class (inherits nn.Module)
# Step 2: Define layers in __init__
# Step 3: Define the forward pass in forward()

class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()  # MUST call parent __init__
        
        # Define layers (registered as model parameters automatically)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # Define how data flows through the network
        x = F.relu(self.fc1(x))       # Linear → ReLU
        x = self.dropout(x)            # Dropout (only active during training)
        x = F.relu(self.fc2(x))       # Linear → ReLU
        x = self.fc3(x)               # Final linear (no activation — loss function handles it)
        return x

# Create an instance
model = SimpleNet(input_dim=20, hidden_dim=64, output_dim=3)
print(model)

# Test with dummy data
x = torch.randn(8, 20)   # Batch of 8, 20 features each
output = model(x)         # Calls forward() automatically — NEVER call model.forward(x)
print(f"\nInput: {x.shape} → Output: {output.shape}")

SimpleNet(
  (fc1): Linear(in_features=20, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

Input: torch.Size([8, 20]) → Output: torch.Size([8, 3])


In [18]:
# nn.Sequential — shorthand for simple models (no branching)
# You'll see this for quick prototypes and as sub-modules inside larger models

model_seq = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

print(model_seq)
print(f"\nOutput: {model_seq(torch.randn(8, 20)).shape}")

# Inspecting model parameters
total_params = sum(p.numel() for p in model_seq.parameters())
trainable = sum(p.numel() for p in model_seq.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

Sequential(
  (0): Linear(in_features=20, out_features=64, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=64, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=3, bias=True)
)

Output: torch.Size([8, 3])

Total parameters:     5,699
Trainable parameters: 5,699


## 6. Loss Functions and Optimizers

The loss function measures how wrong the model is. The optimizer updates weights to reduce the loss.

In [19]:
# Loss functions — choose based on your task

# Classification (multi-class): CrossEntropyLoss
# Takes RAW logits (no softmax needed!) and integer class labels
criterion_ce = nn.CrossEntropyLoss()
logits = torch.randn(8, 3)          # batch of 8, 3 classes (raw scores)
labels = torch.tensor([0, 2, 1, 0, 2, 1, 0, 1])  # ground truth (integers)
loss_ce = criterion_ce(logits, labels)
print(f"CrossEntropy loss: {loss_ce.item():.4f}")

# Classification (binary): BCEWithLogitsLoss
# Takes raw logits and float labels (0.0 or 1.0)
criterion_bce = nn.BCEWithLogitsLoss()
logits_bin = torch.randn(8)
labels_bin = torch.tensor([1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0])
loss_bce = criterion_bce(logits_bin, labels_bin)
print(f"BCE loss:          {loss_bce.item():.4f}")

# Regression: MSELoss (mean squared error)
criterion_mse = nn.MSELoss()
predictions = torch.randn(8)
targets = torch.randn(8)
loss_mse = criterion_mse(predictions, targets)
print(f"MSE loss:          {loss_mse.item():.4f}")

# .item() extracts a Python number from a 0-dim tensor
# You'll see loss.item() everywhere — for logging the loss value

CrossEntropy loss: 1.2681
BCE loss:          0.6766
MSE loss:          1.2828


In [20]:
# Optimizers — update model weights to minimize the loss

model = SimpleNet(20, 64, 3)

# Adam — the default choice for most projects
optimizer = optim.Adam(model.parameters(), lr=0.001)

# SGD — simpler, sometimes better for specific tasks
# optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# AdamW — Adam with proper weight decay (preferred for transformers)
# optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Key parameters:
# lr (learning rate) — how big each update step is
# weight_decay       — L2 regularization (prevents overfitting)
# momentum (SGD)     — accelerates convergence

print("Optimizer:", optimizer)
print("\nParameter groups:", len(optimizer.param_groups))
print("Learning rate:   ", optimizer.param_groups[0]['lr'])

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

Parameter groups: 1
Learning rate:    0.001


## 7. The Training Loop

**The heart of every PyTorch project.** Once you understand this pattern, you can read any training script.

In [21]:
# The 5-step training loop — memorize this pattern

# Setup
model = SimpleNet(20, 64, 3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Fake data
X_train = torch.randn(200, 20)
y_train = torch.randint(0, 3, (200,))

# Training loop
num_epochs = 5

for epoch in range(num_epochs):
    model.train()  # Set model to training mode (enables dropout, batchnorm training behavior)
    
    # In practice, you'd loop over batches from a DataLoader here
    # For simplicity, we'll use the full dataset
    
    # === THE 5 STEPS (always in this order) ===
    
    # Step 1: Forward pass — compute predictions
    outputs = model(X_train)
    
    # Step 2: Compute loss
    loss = criterion(outputs, y_train)
    
    # Step 3: Zero gradients (they accumulate by default!)
    optimizer.zero_grad()
    
    # Step 4: Backward pass — compute gradients
    loss.backward()
    
    # Step 5: Update weights
    optimizer.step()
    
    # Logging
    _, predicted = torch.max(outputs, dim=1)   # Get predicted class
    accuracy = (predicted == y_train).float().mean()
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {loss.item():.4f} Acc: {accuracy.item():.2%}")

Epoch [1/5] Loss: 1.0997 Acc: 32.00%
Epoch [2/5] Loss: 1.0950 Acc: 40.50%
Epoch [3/5] Loss: 1.0936 Acc: 39.50%
Epoch [4/5] Loss: 1.0913 Acc: 39.50%
Epoch [5/5] Loss: 1.0871 Acc: 37.50%


In [22]:
# The evaluation loop — always paired with training

model.eval()  # Set to eval mode (disables dropout, uses running stats for batchnorm)

X_test = torch.randn(50, 20)
y_test = torch.randint(0, 3, (50,))

with torch.no_grad():  # No gradient tracking needed for evaluation
    outputs = model(X_test)
    loss = criterion(outputs, y_test)
    _, predicted = torch.max(outputs, dim=1)
    accuracy = (predicted == y_test).float().mean()

print(f"Test Loss: {loss.item():.4f}")
print(f"Test Acc:  {accuracy.item():.2%}")

# KEY POINTS:
# model.train() → model.eval()  — MUST switch between modes
# torch.no_grad()               — saves memory, speeds up eval
# These two are the most common bugs when eval results look wrong

Test Loss: 1.1085
Test Acc:  30.00%


## 8. Data Loading — `Dataset` and `DataLoader`

In real projects, data is too large to fit in memory at once. PyTorch's data pipeline handles batching, shuffling, and parallel loading.

In [23]:
# Custom Dataset — the standard pattern for loading your own data
# Must implement: __len__ and __getitem__

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create dataset from numpy arrays (as if loaded from pandas)
np.random.seed(42)
X = np.random.randn(100, 10).astype(np.float32)
y = np.random.randint(0, 3, 100)

dataset = TabularDataset(X, y)
print(f"Dataset size: {len(dataset)}")
print(f"Single item:  features={dataset[0][0].shape}, label={dataset[0][1]}")

Dataset size: 100
Single item:  features=torch.Size([10]), label=2


In [24]:
# DataLoader — wraps a Dataset to provide batching, shuffling, and parallel loading

train_loader = DataLoader(
    dataset,
    batch_size=16,       # Number of samples per batch
    shuffle=True,        # Shuffle every epoch (True for training, False for eval)
    # num_workers=4,     # Parallel data loading (set >0 on Linux/Mac for speedup)
    drop_last=False      # Whether to drop the last incomplete batch
)

# Iterate over batches — this is how the training loop really works
for batch_idx, (features, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx}: features={features.shape}, labels={labels.shape}")
    if batch_idx >= 2:
        print("...")
        break

print(f"\nTotal batches per epoch: {len(train_loader)}")

# TensorDataset — quick alternative when data is already tensors
from torch.utils.data import TensorDataset

quick_dataset = TensorDataset(
    torch.randn(100, 10),
    torch.randint(0, 3, (100,))
)
quick_loader = DataLoader(quick_dataset, batch_size=32, shuffle=True)

Batch 0: features=torch.Size([16, 10]), labels=torch.Size([16])
Batch 1: features=torch.Size([16, 10]), labels=torch.Size([16])
Batch 2: features=torch.Size([16, 10]), labels=torch.Size([16])
...

Total batches per epoch: 7


In [25]:
# COMPLETE training loop with DataLoader — the REAL production pattern

model = SimpleNet(10, 32, 3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_features, batch_labels in train_loader:
        # The 5 steps, now inside the batch loop
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate metrics
        running_loss += loss.item() * batch_features.size(0)
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == batch_labels).sum().item()
        total += batch_labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2%}")

Epoch [1/3] Loss: 1.0990 Acc: 39.00%
Epoch [2/3] Loss: 1.0938 Acc: 40.00%
Epoch [3/3] Loss: 1.0914 Acc: 45.00%


## 9. Saving and Loading Models

Two approaches: save the whole model, or save just the weights (state_dict). The state_dict approach is standard.

In [26]:
# state_dict is a Python dict mapping layer names → weight tensors
print("State dict keys:")
for key, value in model.state_dict().items():
    print(f"  {key:20s} → {value.shape}")

# SAVE — recommended approach (just the weights)
# torch.save(model.state_dict(), 'model_weights.pth')

# LOAD — must create model first, then load weights into it
# model = SimpleNet(10, 32, 3)
# model.load_state_dict(torch.load('model_weights.pth'))
# model.eval()  # Always switch to eval after loading for inference

# Save EVERYTHING (for resuming training)
# checkpoint = {
#     'epoch': epoch,
#     'model_state_dict': model.state_dict(),
#     'optimizer_state_dict': optimizer.state_dict(),
#     'loss': loss.item(),
# }
# torch.save(checkpoint, 'checkpoint.pth')

# Load checkpoint
# checkpoint = torch.load('checkpoint.pth')
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print("\n(Save/load commands commented out to avoid file creation)")

State dict keys:
  fc1.weight           → torch.Size([32, 10])
  fc1.bias             → torch.Size([32])
  fc2.weight           → torch.Size([32, 32])
  fc2.bias             → torch.Size([32])
  fc3.weight           → torch.Size([3, 32])
  fc3.bias             → torch.Size([3])

(Save/load commands commented out to avoid file creation)


## 10. GPU / Device Management

PyTorch code is full of `.to(device)` and `.cuda()` calls. Even if you run on CPU, you need to understand the pattern to read the code.

In [27]:
# THE standard device setup — you'll see this at the top of every PyTorch script

# Pattern 1: CUDA only
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Pattern 2: Apple Silicon support (newer code)
# device = torch.device('cuda' if torch.cuda.is_available()
#                        else 'mps' if torch.backends.mps.is_available()
#                        else 'cpu')

# Simplified for this notebook (CPU)
device = torch.device('cpu')
print(f"Using device: {device}")

# Move model to device — MUST do this before training
model = SimpleNet(10, 32, 3).to(device)

# Move data to device — MUST match the device of the model
x = torch.randn(8, 10).to(device)
y = torch.randint(0, 3, (8,)).to(device)

output = model(x)
print(f"Output device: {output.device}")

# Common patterns you'll see:
# model = MyModel().to(device)              — move model to GPU
# model = MyModel().cuda()                  — shorthand (GPU only, less flexible)
# inputs = inputs.to(device)               — move batch to GPU in training loop
# tensor.cpu().numpy()                     — move back to CPU for numpy conversion
# tensor.detach().cpu().numpy()            — the FULL safe conversion pattern

Using device: cpu
Output device: cpu


## 11. Common Architectures — CNN and RNN Patterns

You don't need to memorize these, but you should recognize the structure when reading code.

In [28]:
# CNN — Convolutional Neural Network (for images)
# The classic pattern: Conv → ReLU → Pool → ... → Flatten → Linear

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Convolutional layers (extract spatial features)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Halves spatial dimensions
        
        # Fully connected layers (classify based on features)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)   # After 2 pools: 28→14→7
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        # x shape: (batch, 1, 28, 28) for MNIST
        x = self.pool(F.relu(self.conv1(x)))    # → (batch, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))    # → (batch, 32, 7, 7)
        x = x.flatten(1)                         # → (batch, 32*7*7) = (batch, 1568)
        x = self.dropout(F.relu(self.fc1(x)))   # → (batch, 128)
        x = self.fc2(x)                          # → (batch, 10)
        return x

cnn = SimpleCNN()
# Test with MNIST-like input: 4 images, 1 channel, 28x28
dummy_images = torch.randn(4, 1, 28, 28)
print("CNN output:", cnn(dummy_images).shape)   # (4, 10)
print(f"CNN params: {sum(p.numel() for p in cnn.parameters()):,}")

CNN output: torch.Size([4, 10])
CNN params: 206,922


In [29]:
# RNN / LSTM — for sequential data (text, time series)

class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        # x shape: (batch, seq_len) — integer token IDs
        x = self.embedding(x)          # → (batch, seq_len, embed_dim)
        output, (hidden, cell) = self.lstm(x)  # output: (batch, seq_len, hidden)
        # Take the last timestep's output
        last = output[:, -1, :]         # → (batch, hidden_dim)
        x = self.fc(last)              # → (batch, num_classes)
        return x

lstm = SimpleLSTM(vocab_size=5000, embed_dim=128, hidden_dim=64, num_classes=2)
dummy_text = torch.randint(0, 5000, (8, 50))  # batch of 8 sequences, length 50
print("LSTM output:", lstm(dummy_text).shape)  # (8, 2)

# Key LSTM things to know:
# batch_first=True — makes input shape (batch, seq, features) instead of (seq, batch, features)
# Returns: output (all timesteps), (hidden_state, cell_state)
# hidden[-1] or output[:, -1, :] — last timestep (common for classification)

LSTM output: torch.Size([8, 2])


In [30]:
# Transformer basics — the architecture behind GPT, BERT, and modern AI
# PyTorch provides nn.TransformerEncoder for building transformer models

class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Embedding(512, d_model)  # Positional encoding
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_encoding(positions)
        x = self.transformer(x)
        x = x.mean(dim=1)    # Average pool over sequence (CLS-like)
        return self.fc(x)

transformer = SimpleTransformer(vocab_size=5000, d_model=64, nhead=4,
                                 num_layers=2, num_classes=2)
dummy_input = torch.randint(0, 5000, (8, 30))
print("Transformer output:", transformer(dummy_input).shape)
print(f"Transformer params: {sum(p.numel() for p in transformer.parameters()):,}")

Transformer output: torch.Size([8, 2])
Transformer params: 915,202


## 12. Learning Rate Schedulers and Gradient Clipping

Two techniques you'll see in virtually every serious training script.

In [31]:
# Learning rate schedulers — gradually adjust the learning rate during training

model = SimpleNet(10, 32, 3)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# StepLR — reduce LR by a factor every N epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# CosineAnnealingLR — smooth cosine decay (very popular in modern training)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# ReduceLROnPlateau — reduce when metric stops improving
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

# Usage in training loop:
lrs = []
for epoch in range(30):
    lrs.append(optimizer.param_groups[0]['lr'])
    # ... training code ...
    scheduler.step()  # Called AFTER optimizer.step(), once per epoch

print("LR schedule (first 15 epochs):", [f"{lr:.4f}" for lr in lrs[:15]])

LR schedule (first 15 epochs): ['0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010']


C:\Users\Windows11\AppData\Local\Temp\ipykernel_4268\2157146715.py:20: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()  # Called AFTER optimizer.step(), once per epoch


In [32]:
# Gradient clipping — prevents exploding gradients (essential for RNNs/Transformers)
# You'll see this in the training loop, between loss.backward() and optimizer.step()

model = SimpleNet(10, 32, 3)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

x = torch.randn(8, 10)
y = torch.randint(0, 3, (8,))

# Forward + backward
output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()

# Clip gradients BEFORE optimizer.step()
max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

optimizer.step()
print(f"Gradient clipping with max_norm={max_norm} applied")

Gradient clipping with max_norm=1.0 applied


## 13. Real Patterns You'll See in AI Code

Complete, realistic patterns pulled from actual deep learning projects.

In [33]:
# Pattern 1: Transfer learning / Fine-tuning a pretrained model
# THE most common pattern in modern AI — don't train from scratch, adapt a pretrained model

# Pseudocode for how it looks with torchvision:
# import torchvision.models as models
# 
# model = models.resnet18(pretrained=True)
# 
# # Freeze all layers (don't train them)
# for param in model.parameters():
#     param.requires_grad = False
# 
# # Replace the final layer for your task (this one IS trainable)
# model.fc = nn.Linear(model.fc.in_features, num_classes)
# 
# # Only optimize the new layer
# optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Let's demonstrate the freeze/unfreeze pattern:
model = SimpleNet(10, 32, 3)

# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Unfreeze just the last layer
for param in model.fc3.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable} / {total} parameters ({trainable/total:.1%})")

Trainable: 99 / 1507 parameters (6.6%)


In [34]:
# Pattern 2: Custom loss function
# Sometimes the built-in losses aren't enough — you'll see custom losses in research code

class FocalLoss(nn.Module):
    """Focal Loss — down-weights easy examples, focuses on hard ones.
    Used in object detection (RetinaNet) and imbalanced classification."""
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# Usage — drop-in replacement for nn.CrossEntropyLoss
criterion = FocalLoss(gamma=2.0)
logits = torch.randn(8, 3)
labels = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1])
loss = criterion(logits, labels)
print(f"Focal Loss: {loss.item():.4f}")

Focal Loss: 0.5616


In [35]:
# Pattern 3: Early stopping — stop training when validation loss stops improving

class EarlyStopping:
    """Stop training when validation loss doesn't improve for `patience` epochs."""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

# Usage in training loop:
early_stopping = EarlyStopping(patience=3)

# Simulate validation losses
val_losses = [1.0, 0.8, 0.7, 0.72, 0.73, 0.74, 0.75]
for epoch, val_loss in enumerate(val_losses):
    early_stopping(val_loss)
    status = "STOP!" if early_stopping.should_stop else "continue"
    print(f"Epoch {epoch+1}: val_loss={val_loss:.2f} patience={early_stopping.counter} → {status}")

Epoch 1: val_loss=1.00 patience=0 → continue
Epoch 2: val_loss=0.80 patience=0 → continue
Epoch 3: val_loss=0.70 patience=0 → continue
Epoch 4: val_loss=0.72 patience=1 → continue
Epoch 5: val_loss=0.73 patience=2 → continue
Epoch 6: val_loss=0.74 patience=3 → STOP!
Epoch 7: val_loss=0.75 patience=4 → STOP!


In [36]:
# Pattern 4: Weight initialization
# Most layers auto-initialize, but you'll see manual init in research code

def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)   # Good for sigmoid/tanh
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight)    # Good for ReLU
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0, std=0.02)

model = SimpleNet(10, 32, 3)
model.apply(init_weights)  # Recursively applies to all submodules
print("Weights initialized")

Weights initialized


In [37]:
# Pattern 5: Reproducibility — setting seeds for consistent results
# You'll see this at the top of every serious experiment

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    # For full determinism (slower):
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

set_seed(42)
print("Random after seed:", torch.randn(3))

set_seed(42)  # Same seed → same result
print("Same seed again: ", torch.randn(3))

Random after seed: 

tensor([0.3367, 0.1288, 0.2345])
Same seed again:  tensor([0.3367, 0.1288, 0.2345])


In [38]:
# Pattern 6: Mixed precision training (AMP)
# Uses float16 for speed while keeping float32 where needed for accuracy
# You'll see this in any GPU training script for large models

# from torch.cuda.amp import autocast, GradScaler
#
# scaler = GradScaler()
#
# for batch in train_loader:
#     optimizer.zero_grad()
#
#     with autocast():                    # Runs in mixed precision
#         output = model(inputs)
#         loss = criterion(output, targets)
#
#     scaler.scale(loss).backward()       # Scale loss to prevent underflow
#     scaler.step(optimizer)              # Unscale gradients and step
#     scaler.update()                     # Update the scale factor

print("Mixed precision training (AMP) — see code comments above")
print("Requires GPU, so shown as pseudocode")

Mixed precision training (AMP) — see code comments above
Requires GPU, so shown as pseudocode


## Quick Reference: NumPy vs PyTorch

| Concept | NumPy | PyTorch | Notes |
|---|---|---|---|
| Create array | `np.array([1,2,3])` | `torch.tensor([1,2,3])` | |
| Zeros | `np.zeros((3,4))` | `torch.zeros(3,4)` | No tuple needed in PyTorch |
| Random normal | `np.random.randn(3,4)` | `torch.randn(3,4)` | |
| Shape | `a.shape` | `t.shape` or `t.size()` | |
| Reshape | `a.reshape(3,4)` | `t.reshape(3,4)` or `t.view(3,4)` | `.view()` is more common |
| Axis/dim | `axis=0` | `dim=0` | Same meaning |
| Type cast | `a.astype('float32')` | `t.float()` or `t.to(torch.float32)` | |
| Concatenate | `np.concatenate` | `torch.cat` | |
| Clip | `np.clip(a, 0, 1)` | `torch.clamp(t, 0, 1)` | |
| Matrix multiply | `a @ b` | `t1 @ t2` | |
| To NumPy | — | `t.detach().cpu().numpy()` | Full safe pattern |
| From NumPy | — | `torch.from_numpy(a)` | Shares memory |

## The Essential PyTorch Vocabulary

| Term | What it means |
|---|---|
| **Tensor** | N-dimensional array (like np.ndarray but with GPU + autograd) |
| **requires_grad** | Track operations for backprop (True for model weights) |
| **backward()** | Compute gradients (derivatives) |
| **optimizer.step()** | Update weights using gradients |
| **model.train()** | Enable dropout/batchnorm training behavior |
| **model.eval()** | Disable dropout, use running stats for batchnorm |
| **torch.no_grad()** | Context manager that disables gradient tracking |
| **.detach()** | Remove tensor from computation graph |
| **.to(device)** | Move tensor/model to CPU or GPU |
| **state_dict** | Dictionary of model weights (for saving/loading) |
| **epoch** | One complete pass through the training data |
| **batch** | Subset of data processed at once |
| **loss.item()** | Extract Python number from a scalar tensor |
| **nn.Module** | Base class for all neural networks |
| **forward()** | Defines data flow through the network |